# Conway's Game of Life

This notebook implements Conway's Game of Life in pure Python and compares it against an optimized NumPy version. A visualization of the grid evolving over time is also provided.

In [1]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# To ensure matplotlib animations display inline:
%matplotlib inline

## Pure Python Implementation (Slide 28)

In [2]:
def update(lattice):
    box_length = len(lattice) - 2
    lattice_new = [[0 for _ in range(box_length + 2)] for _ in range(box_length + 2)]
    for i in range(1, box_length + 1):
        for j in range(1, box_length + 1):
            lattice_new[i][j] = update_rule(lattice, i, j)
    return lattice_new

def update_rule(lattice, i, j):
    n_neigh = lattice[i + 1][j] + lattice[i][j + 1] + lattice[i + 1][j + 1] + \
              lattice[i + 1][j - 1] + lattice[i - 1][j] + lattice[i][j - 1] + \
              lattice[i - 1][j + 1] + lattice[i - 1][j - 1]
    if (lattice[i][j] == 1) and (n_neigh in [2, 3]):
        return 1
    elif lattice[i][j] == 1:
        return 0
    elif (lattice[i][j] == 0) and (n_neigh == 3):
        return 1
    else:
        return 0

## Initialization (Cross, Box Size 300)

In [3]:
N = 300
lattice = [[0 for _ in range(N + 2)] for _ in range(N + 2)]

# Draw a cross in the center
mid = N // 2 + 1
for i in range(1, N + 1):
    lattice[mid][i] = 1
    lattice[i][mid] = 1

initial_lattice_py = copy.deepcopy(lattice)
initial_lattice_np = np.array(lattice)

## Optimized NumPy Implementation

In [4]:
def update_numpy(lattice):
    lattice_new = np.zeros_like(lattice)
    
    n_neigh = (
        lattice[2:, 1:-1] + 
        lattice[1:-1, 2:] + 
        lattice[2:, 2:] + 
        lattice[2:, :-2] + 
        lattice[:-2, 1:-1] + 
        lattice[1:-1, :-2] + 
        lattice[:-2, 2:] + 
        lattice[:-2, :-2]
    )
    
    center = lattice[1:-1, 1:-1]
    stay_alive = (center == 1) & ((n_neigh == 2) | (n_neigh == 3))
    become_alive = (center == 0) & (n_neigh == 3)
    
    lattice_new[1:-1, 1:-1][stay_alive | become_alive] = 1

    return lattice_new

## Performance Comparison (300 Updates)

In [5]:
updates = 300
# Pure Python
current_lattice_py = copy.deepcopy(initial_lattice_py)
start_time = time.time()
for _ in range(updates):
    current_lattice_py = update(current_lattice_py)
py_duration = time.time() - start_time
print(f"Pure Python time for {updates} updates: {py_duration:.4f} seconds")

# NumPy
current_lattice_np = initial_lattice_np.copy()
start_time = time.time()
for _ in range(updates):
    current_lattice_np = update_numpy(current_lattice_np)
np_duration = time.time() - start_time
print(f"NumPy time for {updates} updates: {np_duration:.4f} seconds")
print(f"Speedup: {py_duration / np_duration:.2f}x")

Pure Python time for 300 updates: 34.6752 seconds
NumPy time for 300 updates: 0.8464 seconds
Speedup: 40.97x


## Generate GIF of the Lattice (300 Frames)

In [6]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.axis('off')
grid_display = ax.imshow(initial_lattice_np[1:-1, 1:-1], cmap='binary')

def animate(frame):
    global current_lattice_np
    current_lattice_np = update_numpy(current_lattice_np)
    grid_display.set_data(current_lattice_np[1:-1, 1:-1])
    return [grid_display]

# Re-initialize before animation
current_lattice_np = initial_lattice_np.copy()

anim = animation.FuncAnimation(
    fig, animate, frames=300, interval=50, blit=True
)

gif_path = 'game_of_life.gif'
anim.save(gif_path, writer='pillow')
plt.close(fig)

print(f"GIF saved as {gif_path}")
HTML(f'<img src="{gif_path}">')

GIF saved as game_of_life.gif
